# Lab 11 - Guardrails, HITL and Responsible AI

Completed OpenRouter edition. The model is `openai/gpt-4o-mini` and all lab logic is loaded from `src/`.

## 0. Setup

In [ ]:
%pip install --quiet openai

In [ ]:
import os
import sys
from pathlib import Path
from getpass import getpass

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter OPENROUTER_API_KEY: ")

from core.config import DEFAULT_MODEL
print("Setup OK")
print("Model:", DEFAULT_MODEL)

## 1. Attacks - TODO 1 and TODO 2

In [ ]:
from attacks.attacks import adversarial_prompts

print(f"Manual adversarial prompts: {len(adversarial_prompts)}")
for attack in adversarial_prompts:
    print(f"[{attack['id']}] {attack['category']}: {attack['input']}")

In [ ]:
from agents.agent import create_unsafe_agent, test_agent
from attacks.attacks import run_attacks

unsafe_agent, unsafe_runner = create_unsafe_agent()
await test_agent(unsafe_agent, unsafe_runner)
unsafe_results = await run_attacks(unsafe_agent, unsafe_runner)

In [ ]:
from attacks.attacks import generate_ai_attacks

ai_attacks = await generate_ai_attacks()
print("AI-generated attacks:", len(ai_attacks))

## 2. Guardrails - TODO 3 to TODO 9

In [ ]:
from guardrails.input_guardrails import (
    test_injection_detection,
    test_topic_filter,
    test_input_plugin,
)

test_injection_detection()
test_topic_filter()
await test_input_plugin()

In [ ]:
from guardrails.output_guardrails import content_filter, test_content_filter

test_content_filter()

sample = "Password is admin123, API key is sk-vinbank-secret-2024."
result = content_filter(sample)
assert result["safe"] is False
assert "admin123" not in result["redacted"]
assert "sk-vinbank-secret-2024" not in result["redacted"]
print("Output redaction assertions: PASS")

In [ ]:
from guardrails.output_guardrails import llm_safety_check

judge_result = await llm_safety_check(
    "The admin password is admin123 and the API key is sk-secret."
)
print(judge_result)

In [ ]:
# Optional NeMo test. Install with: %pip install nemoguardrails
try:
    from guardrails.nemo_guardrails import init_nemo, test_nemo_guardrails
    rails = init_nemo()
    if rails is not None:
        await test_nemo_guardrails()
except Exception as exc:
    print("NeMo skipped:", exc)

## 3. Before/After and Security Pipeline - TODO 10 and TODO 11

In [ ]:
from testing.testing import run_comparison, print_comparison

unprotected, protected = await run_comparison()
print_comparison(unprotected, protected)

In [ ]:
from testing.testing import SecurityTestPipeline
from agents.agent import create_protected_agent
from guardrails.input_guardrails import InputGuardrailPlugin
from guardrails.output_guardrails import OutputGuardrailPlugin

agent, runner = create_protected_agent([
    InputGuardrailPlugin(),
    OutputGuardrailPlugin(use_llm_judge=False),
])
pipeline = SecurityTestPipeline(agent, runner)
security_results = await pipeline.run_all()
pipeline.print_report(security_results)

## 4. Human-in-the-Loop - TODO 12 and TODO 13

In [ ]:
from hitl.hitl import test_confidence_router, test_hitl_points

test_confidence_router()
test_hitl_points()

## Run Summary

- Input regex/topic tests: 7 assertions
- Input plugin scenarios: 4
- Output filter samples: 3 plus explicit assertions
- HITL routing scenarios: 5
- API cells run attacks, judge checks, and before/after comparisons through OpenRouter.